# Lab 5: Create and query a MongoDB collection


#### Import libraries

In [4]:
%%capture
%pip install pymongo
%pip install gdown

In [5]:
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

#### 1. PyMongo Configuration and Database Connection

In this cell, we are connecting to a MongoDB database hosted on MongoDB Atlas.

1. URI Setup:

- The uri variable holds the connection string, which includes the username and password required to connect to the MongoDB cluster.

2.
Creating a Client:

- `client = MongoClient(uri, server_api=ServerApi('1'), tlsAllowInvalidCertificates=True)`: This line initializes a new client using `MongoClient`, which establishes a connection to the MongoDB server.

- The parameter `server_api=ServerApi('1')` sets the API version for the connection, and `tlsAllowInvalidCertificates=True` allows the client to connect even if the TLS certificate isn't valid (useful for testing environments).


3. Testing the Connection:

- We use a `try` block to send a ping command (`client.admin.command('ping')`) to verify the connection. If successful, it prints a success message.
- If the connection fails, it catches the exception and displays the error message, helping diagnose connection issues.

In [10]:
uri = f""

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'),
                     tlsAllowInvalidCertificates=True)
# Send a ping to confirm a successful connection

try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    raise e

Pinged your deployment. You successfully connected to MongoDB!


Show the list of databases

In [11]:
client.list_database_names()

['lab4', 'lab5', 'admin', 'local']

Upload `restaurants_collection.txt` file

In [12]:
db = client['lab5']

In [9]:
#db.create_collection('restaurants')

CollectionInvalid: collection restaurants already exists

In [13]:
restaurants = db['restaurants']

In [14]:
from bson.json_util import loads
import gdown

url = 'https://drive.google.com/uc?id=1WWd-vk2gyvy4gPG40MtjENPOYXbpIKey'
output = 'restaurants_collection.txt'
gdown.download(url, output, quiet=False)

# Read records (json)
with open(output) as f:
    data = loads(f.read())

restaurants.delete_many({})
# Add records to the collection
restaurants.insert_many(data);


Downloading...
From: https://drive.google.com/uc?id=1WWd-vk2gyvy4gPG40MtjENPOYXbpIKey
To: /home/turing/JupyterLabTuring/Projetos Pessoais/Witeck/restaurants_collection.txt
100%|██████████| 4.24k/4.24k [00:00<00:00, 4.32MB/s]


## Exercises

### 1. Find all restaurants whose cost is medium. Show the result is the pretty format.

In [17]:
from pprint import pprint
results = restaurants.find({'cost': 'medium'}, ["name", "review", "tag"]) #WRITE YOUR QUERY HERE
for result in results:
  pprint(result)

{'_id': ObjectId('67363b807e488bd5fe0c1371'),
 'name': 'ToorSeafoodrestaurant',
 'review': 4.3,
 'tag': ['seafood', 'expensive']}
{'_id': ObjectId('67363b807e488bd5fe0c1379'),
 'name': 'Mcdownloads',
 'review': 3.9,
 'tag': ['fastfood']}
{'_id': ObjectId('67363b807e488bd5fe0c137a'),
 'name': 'OldNavyHamburgar',
 'review': 4.5,
 'tag': ['hamburger', 'fastfood']}


### 2. Select the name and the number of seats (maxPeople) available of all the restaurants whose review is bigger than 4 and cost is medium or low

In [18]:
results = db.restaurants.find({'review': { "$gt" : 4}, 'cost': {"$in": ["medium", "low"]}}, ["name", "maxPeople"]) 
for result in results:
  pprint(result)

{'_id': ObjectId('67363b807e488bd5fe0c1371'),
 'maxPeople': 100,
 'name': 'ToorSeafoodrestaurant'}
{'_id': ObjectId('67363b807e488bd5fe0c1372'),
 'maxPeople': 50,
 'name': 'PandaParadise'}
{'_id': ObjectId('67363b807e488bd5fe0c1376'),
 'maxPeople': 10,
 'name': 'IlDivinPanino'}
{'_id': ObjectId('67363b807e488bd5fe0c1378'),
 'maxPeople': 15,
 'name': 'Smartbar'}
{'_id': ObjectId('67363b807e488bd5fe0c137a'),
 'maxPeople': 100,
 'name': 'OldNavyHamburgar'}


### 3. Select the name, the phone of the restaurants that can contain more than 5 people and:

  #### a. whose tag contains "italian" or "japanese" and cost is medium or high OR
  #### b. whose tag does not contain neither "italian" nor "japanese", and whose review is higher than 4.5

#### Remove from the output the field _id.

In [20]:
results = db.restaurants.find({
    "$and": [
        { "maxPeople": { "$gt": 5 } },
        { 
            "$or":[
                {"tag" : {"$in": ["italian", "japanese"]}},
                {"tag" : {"$in": ["italian", "japanese"]}, "review" : {"$gt" : 4.5}}
            ]            
        }
    ]
    }, { "_id": 0 })
for result in results:
  pprint(result)

{'contact': {'facebook': 'PandaP', 'phone': '+395487634998'},
 'cost': 'low',
 'location': {'coordinates': [45.0671, 7.6627], 'type': 'Point'},
 'maxPeople': 50,
 'name': 'PandaParadise',
 'orderNeeded': False,
 'review': 4.7,
 'tag': ['chinese', 'japanese']}
{'contact': {'phone': '+390223456245'},
 'cost': 'low',
 'location': {'coordinates': [45.0698, 7.6634], 'type': 'Point'},
 'maxPeople': 10,
 'name': 'Stagione',
 'orderNeeded': False,
 'review': 3.8,
 'tag': ['italian', 'pizza']}
{'contact': {},
 'cost': 'low',
 'location': {'coordinates': [45.0587, 7.6612], 'type': 'Point'},
 'maxPeople': 75,
 'name': 'MishiSushi',
 'orderNeeded': False,
 'review': 3.9,
 'tag': ['japanese', 'unlimitedoffering']}
{'contact': {'phone': '+398772376563'},
 'cost': 'high',
 'location': {'coordinates': [45.0661, 7.6544], 'type': 'Point'},
 'maxPeople': 100,
 'name': 'IlTempo',
 'orderNeeded': True,
 'review': 4.2,
 'tag': ['italian', 'cosy']}


### 4. Calculate the average review of all restaurants

In [23]:
results = db.restaurants.aggregate([
    {
        "$group": { "_id": "$name", 
                    "review_avg": { "$avg" : "$review"}}
    }
])
for result in results:
  print(result['review_avg'])

4.7
3.9
4.6
4.2
4.2
4.5
4.5
3.8
3.9
4.3


### 5. Count the number of restaurants whose review is higher than 4.5 and can contain more than 5 people

In [38]:
results = db.restaurants.aggregate([
    {
        "$match" : {"review": { "$gt": 4.5}, "maxPeople": { "$gt": 5 } }
    },
    {
        "$count" : "count"
    }
])

for result in results:
  print(result['count'])

2


### 6. Find the restaurant in the collection which is nearest to the point [45.0644, 7.6598]. Hint: remember to create the geospatial index.

In [61]:
db.restaurants.create_index({"location.coordinates" : "2dsphere"} )

'location.coordinates_2dsphere'

In [62]:
result = db.restaurants.find_one({
  "location.coordinates": {
    "$near": {
      "$geometry": {
        "type": "Point",
        "coordinates": [45.0644, 7.6598]
      }
    }
}})

# find_one returns one single document
pprint(result)

{'_id': ObjectId('67363b807e488bd5fe0c1376'),
 'contact': {'phone': '+393319416860', 'website': 'ildivinpanino.it'},
 'cost': 'low',
 'location': {'coordinates': [45.0645, 7.6608], 'type': 'Point'},
 'maxPeople': 10,
 'name': 'IlDivinPanino',
 'orderNeeded': False,
 'review': 4.6,
 'tag': ['casual', 'goodforkids']}


### 7. Find how many restaurants in the collection are within 500 meters from the point [45.0623, 7.6627]

In [71]:
results = db.restaurants.find({
  "location.coordinates": {
    "$near": {
      "$geometry": {
        "type": "Point",
        "coordinates": [45.0623, 7.6627]
      },
    "$maxDistance": 500
    }
}})

pprint(len(list(results)))

3


### 8. Add the tag “pizza” to all the restaurants that contain the tag “italian”. If the tag “pizza” is already present, you should not insert it

In [76]:
db.restaurants.update_many(
    { "tag" : { "$all": ["italian"]}},
                          
    {  "$addToSet": { "tag": "pizza" }}) 

# We could, alternativally, filter places that dont have the "pizza" tag and give push

UpdateResult({'n': 2, 'electionId': ObjectId('7fffffff00000000000002dd'), 'opTime': {'ts': Timestamp(1731610600, 39), 't': 733}, 'nModified': 0, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1731610600, 39), 'signature': {'hash': b'\xde7\xae\x87\x02\xdaF~\xe2\x13\xab\x80D\xcd\x02\x1c\x119\xdc\xa9', 'keyId': 7374484853258977318}}, 'operationTime': Timestamp(1731610600, 39), 'updatedExisting': True}, acknowledged=True)

### 9. Decrease the review score of 0.2 for all the restaurants with the tag ‘fastfood’

In [86]:
db.restaurants.update_many(
  { "tag": { "$all": ["fastfood"] } },
  [
    {
      "$set": {
        "review": {
          "$cond": {
            "if": { "$isArray": "$review" },
            "then": {
              "$map": {
                "input": "$review",
                "as": "r",
                "in": { "$subtract": ["$$r", 0.2] }
              }
            },
            "else": { "$subtract": ["$review", 0.2] }
          }
        }
      }
    }
  ]
)


UpdateResult({'n': 2, 'electionId': ObjectId('7fffffff00000000000002dd'), 'opTime': {'ts': Timestamp(1731622814, 41), 't': 733}, 'nModified': 2, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1731622814, 41), 'signature': {'hash': b'K\xea\x02P-J\x1e\xe7\xa2h\x1d\x86Y\x17;C\xc9ia~', 'keyId': 7374484853258977318}}, 'operationTime': Timestamp(1731622814, 41), 'updatedExisting': True}, acknowledged=True)

### 10. For only the restaurants with a review higher than 3, find the tags which appear more than 1 time. For each tag, show how many documents include it.

In [88]:
results = db.restaurants.aggregate([
    {"$match": {"review" :{"$gt": 3}}},
    {"$unwind": "$tag"},
     {"$group":{
         "_id": "$tag",
         "count": {"$sum":1}
     }},
     {"$match": {"count":{"$gt" : 1}}}
]) #WRITE YOUR PIPELINE HERE

for result in results:
  pprint(result)

{'_id': 'italian', 'count': 2}
{'_id': 'pizza', 'count': 2}
{'_id': 'fastfood', 'count': 2}
{'_id': 'japanese', 'count': 2}


### 11. For each cost category, compute the minimum review rate, the maximum review rate, the average review rate and the number of restaurants. Sort the result in descending order according to the number of restaurants in each cost category.

In [94]:
results = db.restaurants.aggregate([
    {"$group": {
        "_id": "$cost",
        "min_review_rate": {"$min": "$review"},
        "max_review_rate": {"$max": "$review"},
        "avg_review_rate": {"$avg": "$review"},
        "num_restaurants": {"$sum": 1},
    }}
]) #WRITE YOUR PIPELINE HERE

for result in results:
  pprint(result)from pprint import pprint

results = db.restaurants.aggregate([
    {
        "$group": {
            "_id": "$cost",
            "min_review_rate": { "$min": "$review" },
            "max_review_rate": { "$max": "$review" },
            "avg_review_rate": { "$avg": "$review" },
            "unique_names": { "$addToSet": "$name" }  # Cria um array de nomes únicos
        }
    },
    {
        "$project": {
            "min_review_rate": 1,
            "max_review_rate": 1,
            "avg_review_rate": 1,
            "distinct_name_count": { "$size": "$unique_names" }  # Conta os valores distintos de "name"
        }
    }
])

for result in results:
    pprint(result)


{'_id': 'medium',
 'avg_review_rate': 4.1,
 'distinct_name_count': 3,
 'max_review_rate': 4.3,
 'min_review_rate': 3.6999999999999997}
{'_id': 'high',
 'avg_review_rate': 4.2,
 'distinct_name_count': 2,
 'max_review_rate': 4.2,
 'min_review_rate': 4.2}
{'_id': 'low',
 'avg_review_rate': 4.3,
 'distinct_name_count': 5,
 'max_review_rate': 4.7,
 'min_review_rate': 3.8}


An alternative in case of existing more than one document per restaurant.

In [96]:
results = db.restaurants.aggregate([
    {
        "$group": {
            "_id": "$cost",
            "min_review_rate": { "$min": "$review" },
            "max_review_rate": { "$max": "$review" },
            "avg_review_rate": { "$avg": "$review" },
            "unique_names": { "$addToSet": "$name" }  # Cria um array de nomes únicos
        }
    },
    {
        "$project": {
            "min_review_rate": 1,
            "max_review_rate": 1,
            "avg_review_rate": 1,
            "distinct_name_count": { "$size": "$unique_names" }  # Conta os valores distintos de "name"
        }
    }
])

for result in results:
    pprint(result)


{'_id': 'low',
 'avg_review_rate': 4.3,
 'distinct_name_count': 5,
 'max_review_rate': 4.7,
 'min_review_rate': 3.8}
{'_id': 'high',
 'avg_review_rate': 4.2,
 'distinct_name_count': 2,
 'max_review_rate': 4.2,
 'min_review_rate': 4.2}
{'_id': 'medium',
 'avg_review_rate': 4.1,
 'distinct_name_count': 3,
 'max_review_rate': 4.3,
 'min_review_rate': 3.6999999999999997}


### 12. Find the median value of maxPeople attribute

In [99]:
from pprint import pprint

results = db.restaurants.aggregate([
    {
        "$group": {
            "_id": None,  # Agrupa todos os documentos em um grupo
            "values": { "$push": "$maxPeople" }  # Coleta todos os valores de "maxPeople" em um array
        }
    },
    {
        "$project": {
            "values": { "$sortArray": { "input": "$values", "sortBy": 1 } },  # Ordena o array
            "median": {
                "$let": {
                    "vars": { 
                        "sortedValues": "$values", 
                        "count": { "$size": "$values" } 
                    },
                    "in": {
                        "$cond": [
                            { "$eq": [ { "$mod": ["$$count", 2] }, 0 ] },  # Se o número de elementos for par
                            {
                                "$avg": [
                                    { "$arrayElemAt": ["$$sortedValues", { "$subtract": [{ "$divide": ["$$count", 2] }, 1] }] },
                                    { "$arrayElemAt": ["$$sortedValues", { "$divide": ["$$count", 2] }] }
                                ]
                            },
                            { "$arrayElemAt": ["$$sortedValues", { "$floor": { "$divide": ["$$count", 2] } }] }
                        ]
                    }
                }
            }
        }
    }
])

for result in results:
    pprint(result)


{'_id': None,
 'median': 42.5,
 'values': [10, 10, 15, 50, 70, 75, 100, 100, 100, 150]}


We can see that MongoDB makes costly operations like count_distinct and median costly to implement too